In [ ]:
import pandas as pd
import plotly.express as px
import os


DATA_FOLDER = "data"
VIS_FOLDER = "vis"
SETS = ['SetA']#, 'SetB', 'SetC', 'SetD']

In [ ]:
def load_and_clean(set_name):
    print(f"Feldolgozás: {set_name}...")
    e_path = os.path.join(DATA_FOLDER, f"SCLDDOS2024_{set_name}_events.csv")
    c_path = os.path.join(DATA_FOLDER, f"SCLDDoS2024_{set_name}_components.csv")
    
    if not os.path.exists(e_path) or not os.path.exists(c_path):
        return None
    
   
    e_df = pd.read_csv(e_path, usecols=['Attack ID', 'Start time', 'End time', 'Type', 'Attack code'])
    c_df = pd.read_csv(c_path, usecols=['Attack ID', 'Packet speed', 'Data speed', 'Avg packet len', 'Source IP count'])
    
    df = e_df.merge(c_df, on='Attack ID', how='left')
    df['Set'] = set_name
    
   
    df['Start time'] = pd.to_datetime(df['Start time'], errors='coerce')
    df['End time'] = pd.to_datetime(df['End time'], errors='coerce')
    df['Duration_sec'] = (df['End time'] - df['Start time']).dt.total_seconds()
    

    df['Type'] = df['Type'].astype('category')
    df['Attack code'] = df['Attack code'].astype('category')
    
    return df

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

def prep_data(df):
    """
    Adatok előkészítése a tanításhoz (Tisztítás, Kódolás, Skálázás)
    """
    df_prep = df.copy()

    # 1. Hiányzó értékek kezelése
    # A numerikus oszlopoknál a hiányzó adatot 0-val helyettesítjük (pl. ha nincs forgalom)
    num_cols = ['Packet speed', 'Data speed', 'Avg packet len', 'Source IP count', 'Duration_sec']
    df_prep[num_cols] = df_prep[num_cols].fillna(0)

    # 2. Kategorikus változók kódolása
    # Az 'Attack code' fontos jellemző, alakítsuk számokká (Label Encoding vagy One-Hot)
    le_attack = LabelEncoder()
    df_prep['Attack_code_enc'] = le_attack.fit_transform(df_prep['Attack code'].astype(str))

    # A célváltozó (Type) kódolása: Normal traffic -> 0, Suspicious -> 1, DDoS -> 2
    le_type = LabelEncoder()
    df_prep['Label'] = le_type.fit_transform(df_prep['Type'].astype(str))

    # 3. Skálázás (Mélytanuláshoz elengedhetetlen)
    # A nagy szórású hálózati mérőszámokat standardizáljuk
    scaler = StandardScaler()
    df_prep[num_cols] = scaler.fit_transform(df_prep[num_cols])

    # 4. Feature-ök kiválasztása a modell számára
    # Kivesszük az ID-kat és az eredeti szöveges mezőket
    features = num_cols + ['Attack_code_enc']
    X = df_prep[features]
    y = df_prep['Label']

    print(f"Adat-előkészítés kész. Feature-ök: {features}")
    return X, y, le_type

In [ ]:
all_dfs = []
for s in SETS:
    data = load_and_clean(s)
    